In [1]:
import pandas as pd

# Load the existing translations
df2_translated = pd.read_csv("/kaggle/input/data-nmt-2/data_nmt_2.csv")

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer_n = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B")
model_n = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

...

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/8.55G [00:00<?, ?B/s]

pytorch_model-00003-of-00003.bin:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [3]:
# Function to fill a model column with translations
def fill_model_column(df, source_column, target_column, tokenizer, model, batch_size=2):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    # set source language for tokenizer
    tokenizer.src_lang = "eng_Latn"
    sentences = df[source_column].tolist()
    translations = []

    # Batch translation for efficiency
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        # Generate translations
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("tur_Latn"),
                max_length=128
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
         # Clear memory between batches
        torch.cuda.empty_cache()
    # Fill the target column
    df[target_column] = translations
    return df

In [4]:
df2_translated = fill_model_column(
    df=df2_translated,
    source_column="source_sentence",
    target_column="model3",
    tokenizer=tokenizer_n,
    model=model_n
    )
df2_translated.to_csv("data_nmt_2_all.csv", index=False)